# EvalHub Native Evaluation

Submits an evaluation job to **EvalHub** using the REST API following the EvalHub REST API described in the RHOAI documentation. The code and explanatory text here are original to this project.

The model under test is the **NeMo Guardrails** endpoint — EvalHub runs the benchmark against it and logs results to MLflow.

Reference: [RHOAI EvalHub docs § 2.7 Submit an evaluation job](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.5)

## Configuration

Everything this notebook needs is set in the next cell. In the workshop
workbench the values arrive as environment variables and you can run it as
is; anywhere else, replace the `EDIT ME` placeholders with your own
endpoint, model and namespace. The cell stops with a clear message if
anything is still unset.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit the values below for YOUR cluster.
#
# Inside the workshop workbench these arrive as environment variables and
# you can run this cell unchanged. Anywhere else — a laptop, a different
# namespace, a MaaS gateway — fill in the placeholders marked EDIT ME.
# ══════════════════════════════════════════════════════════════════════
import os

# Your project/namespace. The workshop chart installs into
# tableschnappi-<your-username>, not plain "tableschnappi".
NAMESPACE = os.environ.get("MLFLOW_WORKSPACE") or "EDIT ME: your-namespace"

# Where the model lives. No trailing /v1 — it is appended where needed.
#   in-cluster: http://<model>-predictor.<ns>.svc.cluster.local:8080
#   MaaS:       https://<maas-host>
MODEL_URL  = os.environ.get("VLLM_URL")   or "EDIT ME: https://your-model-endpoint"
MODEL_NAME = os.environ.get("VLLM_MODEL") or "EDIT ME: your-model-id"

# Only needed for an authenticated gateway such as MaaS.
MODEL_API_KEY = os.environ.get("MODEL_API_KEY", "EMPTY")

# Derived from NAMESPACE unless the workbench told us otherwise.
GUARDRAILS_URL = os.environ.get("GUARDRAILS_URL") or f"http://tableschnappi-guardrails.{NAMESPACE}.svc.cluster.local:80"
ASSISTANT_URL  = os.environ.get("ASSISTANT_URL")  or f"http://tableschnappi-mlflow-proxy.{NAMESPACE}.svc.cluster.local:8000"
MCP_URL        = os.environ.get("MCP_URL")        or f"http://tableschnappi-mcp.{NAMESPACE}.svc.cluster.local:3000/mcp"

MLFLOW_URI        = os.environ.get("MLFLOW_TRACKING_URI") or "EDIT ME: https://mlflow.redhat-ods-applications.svc:8443/mlflow"
MLFLOW_EXPERIMENT = "restaurant-chatbot-traces"

# MLflow on RHOAI authenticates with the pod's service-account token and
# serves a self-signed certificate.
os.environ["MLFLOW_TRACKING_URI"]          = MLFLOW_URI
os.environ["MLFLOW_TRACKING_AUTH"]         = "kubernetes-namespaced"
os.environ["MLFLOW_K8S_INTEGRATION"]       = "true"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
os.environ["OPENAI_API_KEY"]               = MODEL_API_KEY

_unset = [n for n, v in [
    ("NAMESPACE", NAMESPACE), ("MODEL_URL", MODEL_URL),
    ("MODEL_NAME", MODEL_NAME), ("MLFLOW_URI", MLFLOW_URI),
] if str(v).startswith("EDIT ME")]

for n, v in [("NAMESPACE", NAMESPACE), ("MODEL_URL", MODEL_URL), ("MODEL_NAME", MODEL_NAME),
             ("GUARDRAILS_URL", GUARDRAILS_URL), ("ASSISTANT_URL", ASSISTANT_URL),
             ("MLFLOW_URI", MLFLOW_URI)]:
    print(f"{n:16} {v}")

if _unset:
    raise SystemExit(
        "\nStill to fill in: " + ", ".join(_unset) +
        "\nEdit them at the top of this cell and run it again."
    )
print("\nConfiguration OK.")


In [ ]:
import subprocess, sys, warnings, urllib3
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--extra-index-url", "https://pypi.org/simple/",
     "mlflow[kubernetes]>=3.11", "httpx"],
    check=True,
)
warnings.filterwarnings("ignore")
urllib3.disable_warnings()

import os, json, time
from urllib.parse import urlparse
import httpx
import mlflow

def detect_cluster_domain():
    """Auto-detect the OpenShift apps domain."""
    if os.environ.get("CLUSTER_DOMAIN"):
        return os.environ["CLUSTER_DOMAIN"]
    mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if mlflow_uri:
        host = urlparse(mlflow_uri).hostname or ""
        if "." in host:
            return host.split(".", 1)[1]
    try:
        import urllib.request, ssl
        token = open("/var/run/secrets/kubernetes.io/serviceaccount/token").read()
        ctx = ssl.create_default_context(
            cafile="/var/run/secrets/kubernetes.io/serviceaccount/ca.crt"
        )
        req = urllib.request.Request(
            "https://kubernetes.default.svc/apis/config.openshift.io/v1/ingresses/cluster",
            headers={"Authorization": f"Bearer {token}"},
        )
        data = json.loads(urllib.request.urlopen(req, context=ctx, timeout=5).read())
        return data["spec"]["domain"]
    except Exception:
        pass
    return ""

CLUSTER_DOMAIN = detect_cluster_domain()
if not CLUSTER_DOMAIN:
    # The workbench injects CLUSTER_DOMAIN directly now. Off-cluster it is
    # only used to build convenience links, so warn rather than abort —
    # this used to be an assert, which stopped a laptop reader one cell
    # after the CONFIGURATION cell told them a laptop was fine.
    print("CLUSTER_DOMAIN not set — the links printed below will be "
          "incomplete. Set it to your apps domain: "
          "oc get ingresses.config/cluster -o jsonpath='{.spec.domain}'")

# NAMESPACE comes from the configuration cell
# MLFLOW_URI comes from the configuration cell
MLFLOW_EXP    = "restaurant-chatbot-traces"
MODEL_URL     = f"{GUARDRAILS_URL}/v1"  # eval through the guardrails shield
# MODEL_NAME comes from the configuration cell

os.environ["MLFLOW_TRACKING_AUTH"]         = "kubernetes-namespaced"
os.environ["MLFLOW_K8S_INTEGRATION"]       = "true"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXP)

EVALHUB_HOST = subprocess.run(
    ["oc", "get", "route", "evalhub", "-n", NAMESPACE,
     "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True,
).stdout.strip()
EVALHUB_URL  = f"https://{EVALHUB_HOST}"

TOKEN = open("/var/run/secrets/kubernetes.io/serviceaccount/token").read().strip()

print(f"CLUSTER_DOMAIN: {CLUSTER_DOMAIN}")
print(f"EvalHub: {EVALHUB_URL}")
print(f"Model:   {MODEL_URL}")
print(f"MLflow:  {MLFLOW_URI}")

## Step 1 — Health check

Verify EvalHub is reachable (docs § 2.3).

In [ ]:
r = httpx.get(f"{EVALHUB_URL}/api/v1/health",
              headers={"Authorization": f"Bearer {TOKEN}"},
              verify=False)
print("Health:", r.json())

## Step 2 — List available providers

Docs § 2.6: list which evaluation frameworks are registered.

In [ ]:
r = httpx.get(
    f"{EVALHUB_URL}/api/v1/evaluations/providers",
    headers={"Authorization": f"Bearer {TOKEN}", "X-Tenant": NAMESPACE},
    verify=False,
)
providers = r.json().get("items", [])
print(f"Available providers ({len(providers)}):")
for p in providers:
    benchmarks = p.get("benchmarks", [])
    print(f"  {p['resource']['id']:<25} {len(benchmarks)} benchmarks")

## Step 3 — Submit evaluation job

Docs § 2.7: POST to `/api/v1/evaluations/jobs` with `X-Tenant` header.

We use `truthfulqa_mc1` — a small truth-testing benchmark (209 questions, no internet needed for the model call itself).

In [ ]:
job_body = {
    "name": f"sakura-truthfulqa-{int(time.time())}",
    "model": {
        "url":  MODEL_URL,
        "name": MODEL_NAME,
    },
    "benchmarks": [
        {
            "id":          "truthfulqa_mc1",
            "provider_id": "lm_evaluation_harness",
        }
    ],
    "limit": "10",     # run only 10 examples to keep it fast
    "allowOnline": True,
}

r = httpx.post(
    f"{EVALHUB_URL}/api/v1/evaluations/jobs",
    headers={
        "Authorization":  f"Bearer {TOKEN}",
        "Content-Type":   "application/json",
        "X-Tenant":       NAMESPACE,
    },
    json=job_body,
    verify=False,
)
print("HTTP", r.status_code)
resp = r.json()
JOB_ID = resp.get("resource", {}).get("id", "")
print("Job ID:", JOB_ID)
print("State: ", resp.get("status", {}).get("state", "?"))

## Step 4 — Poll for completion

Docs § 2.8: GET `/api/v1/evaluations/jobs/{id}` with `X-Tenant` header.

In [ ]:
print(f"Watching job {JOB_ID}...")
for i in range(60):
    r = httpx.get(
        f"{EVALHUB_URL}/api/v1/evaluations/jobs/{JOB_ID}",
        headers={"Authorization": f"Bearer {TOKEN}", "X-Tenant": NAMESPACE},
        verify=False,
    )
    job = r.json()
    state = job.get("status", {}).get("state", "?")
    msg   = job.get("status", {}).get("message", {}).get("message", "")[:60]
    print(f"  [{i*10:>4}s] {state}  {msg}", end="\r")
    if state in ("completed", "failed", "cancelled"):
        print()
        break
    time.sleep(10)

print("Final state:", state)

## Step 5 — Read results and log to MLflow

In [ ]:
results = job.get("results", {})
benchmarks = results.get("benchmarks", [])

print("=" * 50)
print(f"{'EVALHUB RESULTS':^50}")
print("=" * 50)
for b in benchmarks:
    print(f"\nBenchmark: {b.get('id')} ({b.get('provider_id')})")
    for metric, val in b.get("metrics", {}).items():
        if isinstance(val, (int, float)):
            bar = "█" * int(val * 20) + "░" * (20 - int(val * 20))
            print(f"  {metric:<20} {val:.3f}  {bar}")

# Log to MLflow
with mlflow.start_run(run_name="evalhub-truthfulqa"):
    mlflow.log_params({
        "tool":      "evalhub",
        "benchmark": "truthfulqa_mc1",
        "model":     MODEL_NAME,
        "endpoint":  "nemo-guardrails",
        "limit":     10,
    })
    for b in benchmarks:
        for metric, val in b.get("metrics", {}).items():
            if isinstance(val, (int, float)):
                mlflow.log_metric(f"{b['id']}_{metric}".replace(",", "_"), val)
    run_id = mlflow.active_run().info.run_id

print(f"\nLogged to MLflow: {MLFLOW_URI}/#/experiments/runs/{run_id}?workspace={NAMESPACE}")